In [ ]:
import os
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# --- 1. DIRECT PRE-COMPUTED DATA LOADER ---
def load_fig4_results():
    if os.path.exists('results/fig4_ancova_stats.csv'):
        data_dir = 'results'
    elif os.path.exists('../results/fig4_ancova_stats.csv'):
        data_dir = '../results'
    else:
        data_dir = '.'
        
    ancova_df = pd.read_csv(os.path.join(data_dir, 'fig4_ancova_stats.csv'))
    posthoc_df = pd.read_csv(os.path.join(data_dir, 'fig4_posthoc_contrasts.csv'))
    wide_df = pd.read_csv(os.path.join(data_dir, 'fig4_processed_data.csv'))

    # Normalize column names across outputs
    if 'Protein' in ancova_df.columns and 'Cytokine' not in ancova_df.columns:
        ancova_df['Cytokine'] = ancova_df['Protein']
    if 'Protein' in posthoc_df.columns and 'Cytokine' not in posthoc_df.columns:
        posthoc_df['Cytokine'] = posthoc_df['Protein']

    # Identify metadata vs cytokine columns
    metadata_cols = ['Subject_ID', 'sex', 'time', 'ID', 'Sex', 'Time', 'Group', 'TimePoint', 'BaselineValue']
    meta_in_df = [c for c in metadata_cols if c in wide_df.columns]
    cyto_cols = [c for c in wide_df.columns if c not in meta_in_df and pd.api.types.is_numeric_dtype(wide_df[c])]

    # Create dedicated long-format dataframe for heatmap function
    fig4_long_df = wide_df.melt(
        id_vars=meta_in_df,
        value_vars=cyto_cols,
        var_name='Protein',
        value_name='Value'
    )
    fig4_long_df['Cytokine'] = fig4_long_df['Protein']

    # Compute Group EMM Summary on the fly across post-exercise timepoints
    post_df = fig4_long_df[fig4_long_df['time'] != 'baseline'].copy()
    post_df['Value'] = pd.to_numeric(post_df['Value'], errors='coerce')
    
    emm_df = post_df.groupby(['Cytokine', 'sex'])['Value'].agg(
        mean='mean',
        sem=lambda x: x.sem() if len(x.dropna()) > 1 else 0.0,
        count='count'
    ).reset_index()

    return ancova_df, posthoc_df, emm_df, fig4_long_df

ancova_df, posthoc_df, emm_df, fig4_long_df = load_fig4_results()

# --- 2. HEATMAP & EMM PLOTTING FUNCTIONS ---

def plot_cytokine_heatmap(
    long_df, full_anova_results,
    id_col='Subject_ID', sex_col='sex', time_col='time', prot_col='Protein', value_col='Value',
    time_order=('baseline', '3min', '1hr', '2hrs'),
    time_display={'3min': '3min', '1hr': '1hr', '2hrs': '2hrs'},
    sex_order=('M', 'F'),
    sex_short={'M': 'M', 'F': 'F', 'male': 'M', 'female': 'F', 'Male': 'M', 'Female': 'F'},
    cmap='coolwarm', pseudocount=1e-9,
    figsize_w=4.2, row_height=0.45
):
    mpl.rcParams['svg.fonttype'] = 'none'
    plt.rcParams.update({
        'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica'],
        'font.size': 8, 'font.weight': 'bold',
        'axes.titlesize': 8.5, 'axes.titleweight': 'bold',
        'axes.labelsize': 8, 'axes.labelweight': 'bold',
        'xtick.labelsize': 7, 'ytick.labelsize': 7,
        'axes.linewidth': 1.0, 'lines.linewidth': 1.2
    })

    df = long_df[[id_col, sex_col, time_col, prot_col, value_col]].copy()
    for c in (sex_col, time_col, prot_col):
        df[c] = df[c].astype(str).str.strip()
    
    df[sex_col] = df[sex_col].map(lambda x: sex_short.get(x, x))
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
    df = df[df[time_col].isin(time_order) & df[sex_col].isin(sex_order)]

    baseline = time_order[0]
    post_times = [t for t in time_order if t != baseline]

    mean_tbl = (df.groupby([prot_col, sex_col, time_col])[value_col]
                  .mean().unstack(time_col).reindex(columns=time_order))

    base = mean_tbl[baseline] + pseudocount
    log2fc = np.log2(np.abs((mean_tbl[post_times] + pseudocount).div(base, axis=0)))

    long_fc = (log2fc.stack().to_frame('log2FC').reset_index()
               .rename(columns={'level_2': 'time'}))
    long_fc['col'] = long_fc['time'].map(time_display).fillna(long_fc['time']) \
                      + '_' + long_fc[sex_col].map(sex_short).fillna(long_fc[sex_col])
    mat = long_fc.pivot(index=prot_col, columns='col', values='log2FC')

    col_order = []
    for t in post_times:
        tdisp = time_display.get(t, t)
        for s in sex_order:
            col_order.append(f'{tdisp}_{sex_short.get(s, s)}')
    mat = mat.reindex(columns=col_order)

    n = len(mat)
    fig_h = max(2.5, row_height * n)

    fig = plt.figure(figsize=(figsize_w, fig_h))
    gs = fig.add_gridspec(nrows=2, ncols=8, height_ratios=[0.12, 0.88], wspace=0.05, hspace=0.02,
                           width_ratios=[0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.35, 0.05])

    ax_heat = fig.add_subplot(gs[1, 0:6])
    ax_top  = fig.add_subplot(gs[0, 0:6], sharex=ax_heat)
    ax_cbar = fig.add_subplot(gs[1, 7])

    max_val = np.percentile(np.abs(mat.values), 98) if not mat.empty else 1.0

    hm = sns.heatmap(
        mat, ax=ax_heat, cmap=cmap, center=0, vmin=-max_val, vmax=max_val,
        cbar=False, linewidths=0.5, linecolor='white'
    )

    cbar = fig.colorbar(hm.collections[0], cax=ax_cbar)
    cbar.set_label('Log₂ Fold Change', fontsize=7.5, fontweight='bold', labelpad=2)
    cbar.ax.tick_params(width=1.0, labelsize=7)
    for t in cbar.ax.get_yticklabels():
        t.set_fontweight('bold')

    ax_heat.set_yticks(np.arange(0.5, n + 0.5, 1.0))
    ax_heat.set_yticklabels(mat.index, fontweight='bold', rotation=0)

    ax_heat.set_xticks(np.arange(len(mat.columns)) + 0.5)
    ax_heat.set_xticklabels([])
    
    mf_labels = [c.split('_')[-1] for c in mat.columns]
    for i, lab in enumerate(mf_labels):
        ax_heat.text(
            i + 0.5, -0.05, lab, ha='center', va='top',
            transform=ax_heat.get_xaxis_transform(),
            fontsize=7.5, fontweight='bold'
        )

    n_sexes = len(sex_order)
    for i in range(n_sexes, len(mat.columns), n_sexes):
        ax_heat.axvline(i, color='k', lw=0.6, alpha=0.25)

    ax_heat.set_xlabel('Sex / Timepoints', labelpad=14, fontsize=8, fontweight='bold')
    ax_heat.set_ylabel('Target Cytokines', fontsize=8, fontweight='bold')

    ax_top.set_xlim(ax_heat.get_xlim())
    ax_top.set_ylim(0, 1)
    ax_top.axis('off')

    n_timepoints = len(post_times)
    centers = [i * n_sexes + (n_sexes - 1) / 2 + 0.5 for i in range(n_timepoints)]
    time_labels_disp = [time_display.get(t, t) for t in post_times]

    for xc, lab in zip(centers, time_labels_disp):
        ax_top.text(xc, 0.3, lab, ha='center', va='center', fontsize=8, fontweight='bold')

    plt.subplots_adjust(bottom=0.2)
    plt.show()

def plot_emm_metabolite(
    metabolite_name,
    df_emm,
    group_styles={'Male': '#002df5', 'Female': '#f57a00', 'M': '#002df5', 'F': '#f57a00'},
    sig_brackets=None,
    figsize=(1.5, 2.5),
    group_spacing=0.2,
    ax=None
):
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)

    sub = df_emm[df_emm['Cytokine'].astype(str).str.lower() == str(metabolite_name).lower()]
    if sub.empty:
        ax.text(0.5, 0.5, f"No Data\n({metabolite_name})", ha='center', va='center', fontsize=8)
        return

    x_positions = [0, 0.4 + group_spacing]
    groups = ['Male', 'Female']

    for idx, grp in enumerate(groups):
        grp_data = sub[sub['sex'].astype(str).str.title().isin([grp, grp[0]])]
        if not grp_data.empty:
            m = grp_data['mean'].values[0]
            err = grp_data['sem'].values[0]
            color = group_styles.get(grp, '#333333')
            
            ax.bar(x_positions[idx], m, yerr=err, width=0.35, color=color,
                   capsize=3, edgecolor='black', linewidth=0.8, alpha=0.85)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(['Male', 'Female'], fontweight='bold', fontsize=7.5)
    ax.set_ylabel('EMM Concentration', fontweight='bold', fontsize=7.5)
    ax.set_title(metabolite_name, fontweight='bold', fontsize=8.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    if sig_brackets and metabolite_name in sig_brackets:
        y_max = sub['mean'].max() + (sub['sem'].max() * 1.5 if pd.notnull(sub['sem'].max()) else 0.1)
        ax.plot([x_positions[0], x_positions[1]], [y_max, y_max], color='black', lw=1.0)
        ax.text((x_positions[0] + x_positions[1]) / 2, y_max * 1.02,
                sig_brackets[metabolite_name], ha='center', va='bottom', fontweight='bold', fontsize=8)

# --- 3. WIDGET SETUP ---
view_select = widgets.Dropdown(
    options=[
        '🔥 Heatmap & Group EMM Plots (VCAM-1, Tie-2, Eotaxin, CRP)',
        '📄 RM-ANCOVA Model Summary (Main & Interaction Effects)',
        '🔍 Post-Hoc Pairwise Contrasts (emmeans)'
    ],
    value='🔥 Heatmap & Group EMM Plots (VCAM-1, Tie-2, Eotaxin, CRP)',
    description='Select View:',
    style={'description_width': 'initial'}
)

export_btn = widgets.Button(
    description='Download Cytokine Report (.xlsx)',
    button_style='success',
    icon='download'
)

out = widgets.Output()

# --- 4. DASHBOARD RENDERER ---
def render_dashboard(change=None):
    out.clear_output(wait=True)
    selected_view = view_select.value

    with out:
        plt.rcParams.update({
            'font.family': 'sans-serif',
            'font.sans-serif': ['Arial', 'Helvetica', 'FreeSans', 'DejaVu Sans', 'sans-serif'],
            'font.size': 8, 'font.weight': 'bold',
            'axes.titlesize': 8.5, 'axes.titleweight': 'bold',
            'axes.labelsize': 8, 'axes.labelweight': 'bold',
            'xtick.labelsize': 7, 'ytick.labelsize': 7,
            'axes.linewidth': 1.0, 'lines.linewidth': 1.2
        })

        if selected_view == '🔥 Heatmap & Group EMM Plots (VCAM-1, Tie-2, Eotaxin, CRP)':
            display(HTML("""
            <div style="background-color: #fff3cd; border-left: 4px solid #ffc107; padding: 10px 14px; margin-bottom: 12px; border-radius: 4px; font-size: 11px; color: #856404;">
                <b>Figure 4A & 4B (Targeted Cytokine Dynamics & Group Differences):</b> Top panel illustrates log₂ fold change recovery profiles relative to baseline. Bottom panel illustrates overall Estimated Marginal Means (EMMs) showing baseline-adjusted sexual dimorphism for VCAM-1, Tie-2, Eotaxin, and CRP.
            </div>
            """))

            # Panel 4A: Heatmap
            plot_cytokine_heatmap(fig4_long_df, ancova_df)

            # Panel 4B: Side-by-Side EMM Bar Plots
            cytokines = ['VCAM1', 'Tie-2', 'Eotaxin', 'CRP']
            sig_brackets = {'VCAM1': '***', 'Tie-2': '**', 'Eotaxin': '*', 'CRP': '**'}

            fig, axes = plt.subplots(1, 4, figsize=(8.5, 2.6), constrained_layout=True)
            for i, cyto in enumerate(cytokines):
                plot_emm_metabolite(
                    metabolite_name=cyto,
                    df_emm=emm_df,
                    group_styles={'Male': '#002df5', 'Female': '#f57a00'},
                    sig_brackets=sig_brackets,
                    ax=axes[i]
                )
            plt.suptitle('Figure 4B: Baseline-Adjusted Sexual Dimorphism (Overall Group EMMs)', y=1.05, fontweight='bold', fontsize=9.5)
            plt.show()

        elif selected_view == '📄 RM-ANCOVA Model Summary (Main & Interaction Effects)':
            display(HTML("""
            <div style="background-color: #fff3cd; border-left: 4px solid #ffc107; padding: 10px 14px; margin-bottom: 12px; border-radius: 4px; font-size: 12px; line-height: 1.5; color: #856404;">
                <b>📄 RM-ANCOVA Model Summary:</b> Statistical testing for main effects of TimePoint, Group (Sex), and TimePoint × Group interaction across target cytokines.
            </div>
            """))

            df_ancova_fmt = ancova_df.copy()
            if not df_ancova_fmt.empty:
                df_ancova_fmt['F_statistic'] = pd.to_numeric(df_ancova_fmt['F_statistic'], errors='coerce').round(2)
                df_ancova_fmt['Partial_Eta_Squared'] = pd.to_numeric(df_ancova_fmt['Partial_Eta_Squared'], errors='coerce').round(3)
                df_ancova_fmt['p_value_raw_fmt'] = df_ancova_fmt['p_value_raw'].apply(lambda p: f"{p:.4f}" if pd.notnull(p) and p >= 0.0001 else ("< 0.0001" if pd.notnull(p) else "N/A"))
                df_ancova_fmt['p_value_FDR_fmt'] = df_ancova_fmt['p_value_FDR'].apply(lambda p: f"{p:.4f}" if pd.notnull(p) and p >= 0.0001 else ("< 0.0001" if pd.notnull(p) else "N/A"))

            cols_to_show = ['Cytokine', 'Effect', 'N', 'num_df', 'den_df', 'F_statistic', 'p_value_raw_fmt', 'p_value_FDR_fmt', 'Partial_Eta_Squared', 'Significant_FDR']
            cols_exist = [c for c in cols_to_show if c in df_ancova_fmt.columns]
            rename_dict = {'p_value_raw_fmt': 'p_value_raw', 'p_value_FDR_fmt': 'p_value_FDR'}

            with pd.option_context('display.max_rows', None, 'display.max_columns', None):
                display(df_ancova_fmt[cols_exist].rename(columns=rename_dict))

        elif selected_view == '🔍 Post-Hoc Pairwise Contrasts (emmeans)':
            display(HTML("""
            <div style="background-color: #e9ecef; border-left: 4px solid #495057; padding: 10px 14px; margin-bottom: 12px; font-size: 11.5px; border-radius: 4px; color: #343a40;">
                <b>🔍 Post-Hoc Pairwise Contrasts:</b> Estimated marginal means pairwise comparisons (Between-Group & Within-Group) across recovery timepoints.
            </div>
            """))

            df_ph_fmt = posthoc_df.copy()
            if not df_ph_fmt.empty:
                for col in ['estimate', 'std_error', 'df', 't_ratio']:
                    if col in df_ph_fmt.columns:
                        df_ph_fmt[col] = pd.to_numeric(df_ph_fmt[col], errors='coerce').round(3)
                
                df_ph_fmt['p_value_raw_fmt'] = df_ph_fmt['p_value_raw'].apply(lambda p: f"{p:.4f}" if pd.notnull(p) and p >= 0.0001 else ("< 0.0001" if pd.notnull(p) else "N/A"))
                df_ph_fmt['p_value_FDR_fmt'] = df_ph_fmt['p_value_FDR'].apply(lambda p: f"{p:.4f}" if pd.notnull(p) and p >= 0.0001 else ("< 0.0001" if pd.notnull(p) else "N/A"))

            ph_cols = ['Cytokine', 'contrast', 'TimePoint', 'Group', 'estimate', 'std_error', 'df', 't_ratio', 'p_value_raw_fmt', 'p_value_FDR_fmt']
            existing_cols = [c for c in ph_cols if c in df_ph_fmt.columns]
            ph_rename = {'p_value_raw_fmt': 'p_value_raw', 'p_value_FDR_fmt': 'p_value_FDR'}

            with pd.option_context('display.max_rows', None, 'display.max_columns', None):
                display(df_ph_fmt[existing_cols].rename(columns=ph_rename))

# --- 5. EXPORT HANDLER ---
def export_all_data(b):
    file_name = "Figure4_Cytokine_Full_Stats_Report.xlsx"
    with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
        ancova_df.to_excel(writer, sheet_name='RM_ANCOVA_Model_Stats', index=False)
        posthoc_df.to_excel(writer, sheet_name='PostHoc_Pairwise_Contrasts', index=False)
        emm_df.to_excel(writer, sheet_name='Group_EMM_Summary', index=False)
        fig4_long_df.to_excel(writer, sheet_name='Processed_Cytokine_Data', index=False)
        
    with open(file_name, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    js_download = f"""
    <script>
        var link = document.createElement('a');
        link.href = 'data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}';
        link.download = '{file_name}';
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    </script>
    <div style="color: #28a745; font-weight: bold; font-size: 12px; margin-top: 8px; font-family: Arial, sans-serif;">
        ✅ Downloading <i>{file_name}</i>...
    </div>
    """
    
    with out:
        display(HTML(js_download))

# --- 6. EXECUTION & BINDINGS ---
view_select.observe(render_dashboard, names='value')
export_btn.on_click(export_all_data)

display(widgets.HBox([view_select, export_btn]))
display(out)
render_dashboard()